# 🎙️ Qwen3-TTS — Malayalam LoRA Fine-Tuning (Colab)

**Dataset:** `siyah1/Malayalam-TTS-v2`  
**Base model:** `Qwen/Qwen3-TTS-12Hz-1.7B-Base`  
**Method:** LoRA via PEFT  

> **GPU required** — Runtime → Change runtime type → GPU (T4 is free, A100 is faster)

## Step 0 — Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else '⚠️  No GPU — switch runtime to GPU first!')

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install -q \
    transformers==4.51.3 \
    accelerate \
    peft \
    safetensors \
    datasets \
    huggingface_hub \
    librosa \
    soundfile \
    tensorboard
print('✅ Done')

## Step 2 — Download Model + Extract `qwen_tts` Engine

The `qwen_tts` Python package ships **inside the model repo** alongside the weights.  
We download the full repo and add it to `sys.path` — no separate git clone needed.

In [ ]:
import os, sys
from huggingface_hub import snapshot_download

WORK_DIR  = '/content/ttsfine'
MODEL_DIR = f'{WORK_DIR}/Qwen3-TTS-Base'
FT_DIR    = f'{WORK_DIR}/finetuning'
os.makedirs(WORK_DIR,  exist_ok=True)
os.makedirs(FT_DIR,    exist_ok=True)

# ── Download model weights + code files ──────────────────────────────────────
if not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print('Downloading Qwen3-TTS-12Hz-1.7B-Base …')
    snapshot_download(
        repo_id='Qwen/Qwen3-TTS-12Hz-1.7B-Base',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.msgpack', 'flax_model*'],
    )
    print('✅ Model downloaded.')
else:
    print(f'✅ Model already at {MODEL_DIR}')

# ── Add qwen_tts package (lives inside the model repo) to Python path ────────
#    The model repo ships the qwen_tts/ package alongside the weights.
for p in [MODEL_DIR, FT_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Quick sanity check
try:
    import qwen_tts
    print(f'✅ qwen_tts found at: {qwen_tts.__file__}')
except ModuleNotFoundError:
    # Fallback: the package might be one level deeper inside model dir
    for root, dirs, files in os.walk(MODEL_DIR):
        if 'qwen_tts' in dirs:
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f'✅ qwen_tts found (fallback) at: {root}/qwen_tts')
            break
    else:
        print('⚠️  qwen_tts NOT found — run the cell below to write it manually.')

### Step 2b — Write `dataset.py` into the finetuning dir

In [ ]:
DATASET_CODE = r'''
# coding=utf-8
from typing import Any, List, Tuple, Union
import librosa, numpy as np, torch
from qwen_tts.core.models.configuration_qwen3_tts import Qwen3TTSConfig
from qwen_tts.core.models.modeling_qwen3_tts import mel_spectrogram
from torch.utils.data import Dataset

class TTSDataset(Dataset):
    def __init__(self, data_list, processor, config: Qwen3TTSConfig, lag_num=-1):
        self.data_list = data_list
        self.processor = processor
        self.lag_num   = lag_num
        self.config    = config

    def __len__(self): return len(self.data_list)

    def _load_audio(self, x):
        audio, sr = librosa.load(x, sr=None, mono=True)
        if audio.ndim > 1: audio = np.mean(audio, axis=-1)
        return audio.astype(np.float32), int(sr)

    def _norm_audio(self, audios):
        items = audios if isinstance(audios, list) else [audios]
        out = []
        for a in items:
            if isinstance(a, str): out.append(self._load_audio(a))
            elif isinstance(a, tuple): out.append((a[0].astype(np.float32), int(a[1])))
            else: raise TypeError(f"Unsupported type: {type(a)}")
        return out

    def _build_text(self, text): return f"<|im_start|>assistant\n{text}<|im_end|>\n<|im_start|>assistant\n"
    def _ensure_list(self, x): return x if isinstance(x, list) else [x]

    def _tokenize(self, text):
        t = self.processor(text=text, return_tensors="pt", padding=True)["input_ids"]
        return t.unsqueeze(0) if t.dim() == 1 else t

    @torch.inference_mode()
    def extract_mels(self, audio, sr):
        assert sr == 24000
        return mel_spectrogram(torch.from_numpy(audio).unsqueeze(0),
            n_fft=1024, num_mels=128, sampling_rate=24000,
            hop_size=256, win_size=1024, fmin=0, fmax=12000).transpose(1, 2)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        text_ids = self._tokenize(self._build_text(item["text"]))
        audio_codes = torch.tensor(item["audio_codes"], dtype=torch.long)
        wav, sr = self._norm_audio(self._ensure_list(item["ref_audio"]))[0]
        ref_mel = self.extract_mels(audio=wav, sr=sr)
        return {"text_ids": text_ids[:, :-5], "audio_codes": audio_codes, "ref_mel": ref_mel}

    def collate_fn(self, batch):
        item_length = [b["text_ids"].shape[1] + b["audio_codes"].shape[0] for b in batch]
        max_length  = max(item_length) + 8
        b, t = len(batch), max_length
        input_ids            = torch.zeros((b,t,2),  dtype=torch.long)
        codec_ids            = torch.zeros((b,t,16), dtype=torch.long)
        text_embedding_mask  = torch.zeros((b,t),    dtype=torch.bool)
        codec_embedding_mask = torch.zeros((b,t),    dtype=torch.bool)
        codec_mask           = torch.zeros((b,t),    dtype=torch.bool)
        attention_mask       = torch.zeros((b,t),    dtype=torch.long)
        codec_0_labels       = torch.full((b,t), -100, dtype=torch.long)
        for i, data in enumerate(batch):
            tid = data["text_ids"]; ac0 = data["audio_codes"][:,0]; acs = data["audio_codes"]
            tl = tid.shape[1]; cl = ac0.shape[0]
            input_ids[i,:3,0] = tid[0,:3]
            input_ids[i,3:7,0] = self.config.tts_pad_token_id
            input_ids[i,7,0] = self.config.tts_bos_token_id
            input_ids[i,8:8+tl-3,0] = tid[0,3:]
            input_ids[i,8+tl-3,0] = self.config.tts_eos_token_id
            input_ids[i,8+tl-2:8+tl+cl,0] = self.config.tts_pad_token_id
            text_embedding_mask[i,:8+tl+cl] = True
            input_ids[i,3:8,1] = torch.tensor([
                self.config.talker_config.codec_nothink_id,
                self.config.talker_config.codec_think_bos_id,
                self.config.talker_config.codec_think_eos_id,
                0, self.config.talker_config.codec_pad_id])
            input_ids[i,8:8+tl-3,1] = self.config.talker_config.codec_pad_id
            input_ids[i,8+tl-3,1] = self.config.talker_config.codec_pad_id
            input_ids[i,8+tl-2,1] = self.config.talker_config.codec_bos_id
            input_ids[i,8+tl-1:8+tl-1+cl,1] = ac0
            input_ids[i,8+tl-1+cl,1] = self.config.talker_config.codec_eos_token_id
            codec_0_labels[i,8+tl-1:8+tl-1+cl] = ac0
            codec_0_labels[i,8+tl-1+cl] = self.config.talker_config.codec_eos_token_id
            codec_ids[i,8+tl-1:8+tl-1+cl,:] = acs
            codec_embedding_mask[i,3:8+tl+cl] = True
            codec_embedding_mask[i,6] = False
            codec_mask[i,8+tl-1:8+tl-1+cl] = True
            attention_mask[i,:8+tl+cl] = True
        ref_mels = torch.cat([d["ref_mel"] for d in batch], dim=0)
        return {"input_ids":input_ids, "ref_mels":ref_mels, "attention_mask":attention_mask,
                "text_embedding_mask":text_embedding_mask.unsqueeze(-1),
                "codec_embedding_mask":codec_embedding_mask.unsqueeze(-1),
                "codec_0_labels":codec_0_labels, "codec_ids":codec_ids, "codec_mask":codec_mask}
'''

import os
FT_DIR = '/content/ttsfine/finetuning'
os.makedirs(FT_DIR, exist_ok=True)
with open(f'{FT_DIR}/dataset.py', 'w') as f:
    f.write(DATASET_CODE)
print('✅ dataset.py written to', FT_DIR)

## Step 3 — Download & Prepare Malayalam Dataset

In [ ]:
from datasets import load_dataset
import os, json, soundfile as sf

WORK_DIR   = '/content/ttsfine'
DATA_DIR   = f'{WORK_DIR}/malayalam_data'
WAV_DIR    = f'{DATA_DIR}/wavs'
TRAIN_JSONL = f'{DATA_DIR}/train_with_codes.jsonl'
VAL_JSONL   = f'{DATA_DIR}/val_with_codes.jsonl'
os.makedirs(WAV_DIR, exist_ok=True)

if os.path.exists(TRAIN_JSONL):
    print('✅ Dataset already prepared — skipping.')
else:
    print('Loading siyah1/Malayalam-TTS-v2 from HuggingFace...')
    ds    = load_dataset('siyah1/Malayalam-TTS-v2', split='train')
    print(f'Total samples: {len(ds)}')
    split = ds.train_test_split(test_size=0.05, seed=42)

    for split_name, ssplit in [('train', split['train']), ('val', split['test'])]:
        out_path = TRAIN_JSONL if split_name == 'train' else VAL_JSONL
        with open(out_path, 'w', encoding='utf-8') as fout:
            for idx, row in enumerate(ssplit):
                wav_path = f'{WAV_DIR}/{split_name}_{idx:06d}.wav'
                if not os.path.exists(wav_path):
                    audio = row['audio']
                    sf.write(wav_path, audio['array'], audio['sampling_rate'])
                fout.write(json.dumps({
                    'audio':       wav_path,
                    'ref_audio':   wav_path,
                    'text':        row['text'],
                    'audio_codes': row['audio_codes'],
                    'language':    'Auto',
                }, ensure_ascii=False) + '\n')
        print(f'✅ {split_name}: {len(ssplit)} samples')

print('Dataset ready.')

## Step 4 — Configuration & Imports

> All imports are here, **after** sys.path is guaranteed to be set.

In [ ]:
import sys, os, json, torch

# ── Paths ─────────────────────────────────────────────────────────────────────
WORK_DIR          = '/content/ttsfine'
MODEL_DIR         = f'{WORK_DIR}/Qwen3-TTS-Base'
FT_DIR            = f'{WORK_DIR}/finetuning'
INIT_MODEL_PATH   = MODEL_DIR
OUTPUT_MODEL_PATH = f'{WORK_DIR}/output_malayalam_lora'
TRAIN_JSONL       = f'{WORK_DIR}/malayalam_data/train_with_codes.jsonl'
VAL_JSONL         = f'{WORK_DIR}/malayalam_data/val_with_codes.jsonl'

# ── sys.path (MUST come before any qwen_tts or dataset imports) ───────────────
for p in [MODEL_DIR, FT_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Fallback: search inside MODEL_DIR for qwen_tts package
try:
    import qwen_tts
except ModuleNotFoundError:
    for root, dirs, _ in os.walk(MODEL_DIR):
        if 'qwen_tts' in dirs:
            sys.path.insert(0, root)
            break

# ── Training hparams ──────────────────────────────────────────────────────────
BATCH_SIZE              = 1       # T4=1, A100=2
EVAL_BATCH_SIZE         = 1
GRAD_ACCUMULATION_STEPS = 8       # effective batch = 8
LR                      = 2e-6
NUM_EPOCHS              = 10
START_EPOCH             = 0
SAVE_EVERY              = 1
EVAL_EVERY              = 1
MIXED_PRECISION         = 'bf16'  # 'fp16' on T4, 'bf16' on A100
ATTN_IMPLEMENTATION     = 'sdpa'
SPEAKER_NAME            = 'malayalam_speaker'

# ── LoRA hparams ──────────────────────────────────────────────────────────────
LORA_RANK           = 16
LORA_ALPHA          = 32
LORA_DROPOUT        = 0.05
LORA_BIAS           = 'none'
LORA_TARGET_MODULES = 'q_proj,k_proj,v_proj,o_proj,gate_proj,up_proj,down_proj'
RESUME_ADAPTER      = None   # e.g. 'output_malayalam_lora/checkpoint-epoch-2'

os.makedirs(OUTPUT_MODEL_PATH, exist_ok=True)

# ── Imports ───────────────────────────────────────────────────────────────────
from accelerate import Accelerator
from dataset import TTSDataset
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel
from safetensors.torch import save_file
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoConfig
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')
print('✅ All imports successful.')

## Step 5 — Helper Functions

In [ ]:
target_speaker_embedding = None

def _parse_list(s): return [x.strip() for x in s.split(',') if x.strip()]

def compute_loss(model, batch):
    global target_speaker_embedding
    input_ids            = batch['input_ids']
    codec_ids            = batch['codec_ids']
    ref_mels             = batch['ref_mels']
    text_embedding_mask  = batch['text_embedding_mask']
    codec_embedding_mask = batch['codec_embedding_mask']
    attention_mask       = batch['attention_mask']
    codec_0_labels       = batch['codec_0_labels']
    codec_mask           = batch['codec_mask']

    with torch.no_grad():
        spk_emb = model.speaker_encoder(
            ref_mels.to(dtype=next(model.parameters()).dtype,
                        device=next(model.parameters()).device)
        ).detach()

    if model.training and target_speaker_embedding is None:
        target_speaker_embedding = spk_emb.detach().to('cpu')

    te = model.talker.model.text_embedding(input_ids[:,:,0])
    if hasattr(model.talker, 'text_projection'): te = model.talker.text_projection(te)
    te  = te * text_embedding_mask
    ce  = model.talker.model.codec_embedding(input_ids[:,:,1]) * codec_embedding_mask
    ce[:, 6, :] = spk_emb
    emb = te + ce
    for i in range(1, 16):
        emb = emb + model.talker.code_predictor.get_input_embeddings()[i-1](codec_ids[:,:,i]) * codec_mask.unsqueeze(-1)

    out = model.talker(
        inputs_embeds=emb[:, :-1, :],
        attention_mask=attention_mask[:, :-1],
        labels=None,
        output_hidden_states=True,
    )
    loss0 = torch.nn.functional.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        codec_0_labels[:, 1:].reshape(-1),
        ignore_index=-100,
    )
    hs  = out.hidden_states[0][-1]
    _, loss1 = model.talker.forward_sub_talker_finetune(
        codec_ids[codec_mask], hs[codec_mask[:, 1:]]
    )
    return loss0 + loss1

def evaluate(model, dl, accelerator):
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in dl:
            g = accelerator.gather_for_metrics(compute_loss(model, batch).detach())
            losses.append(g.unsqueeze(0) if g.ndim == 0 else g)
    model.train()
    return torch.cat(losses).mean().item() if losses else None

print('✅ Helper functions ready.')

## Step 6 — Load Model + Apply LoRA

In [ ]:
accelerator = Accelerator(
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    mixed_precision=None if MIXED_PRECISION == 'no' else MIXED_PRECISION,
    log_with='tensorboard',
    project_dir=OUTPUT_MODEL_PATH,
)
print(f'Device: {accelerator.device} | Precision: {MIXED_PRECISION}')

print(f'\nLoading {INIT_MODEL_PATH} ...')
qwen3tts = Qwen3TTSModel.from_pretrained(
    INIT_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    attn_implementation=ATTN_IMPLEMENTATION,
)
config = AutoConfig.from_pretrained(INIT_MODEL_PATH)

if RESUME_ADAPTER:
    model = PeftModel.from_pretrained(qwen3tts.model, RESUME_ADAPTER, is_trainable=True)
else:
    model = get_peft_model(qwen3tts.model, LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        bias=LORA_BIAS, target_modules=_parse_list(LORA_TARGET_MODULES),
        task_type=TaskType.CAUSAL_LM,
    ))

# use_reentrant=False fixes the 'does not require grad' error with LoRA + gradient checkpointing
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': False})
model.print_trainable_parameters()

## Step 7 — Build Dataloaders

In [ ]:
with open(TRAIN_JSONL, 'r', encoding='utf-8') as f:
    train_data = [json.loads(l) for l in f]

ds_train     = TTSDataset(train_data, qwen3tts.processor, config)
train_dl     = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=ds_train.collate_fn)
print(f'Training samples : {len(train_data)}')

val_dl = None
if os.path.exists(VAL_JSONL):
    with open(VAL_JSONL, 'r', encoding='utf-8') as f:
        val_data = [json.loads(l) for l in f]
    ds_val = TTSDataset(val_data, qwen3tts.processor, config)
    val_dl = DataLoader(ds_val, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=ds_val.collate_fn)
    print(f'Validation samples: {len(val_data)}')

## Step 8 — Train 🚀

In [ ]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)

if val_dl:
    model, optimizer, train_dl, val_dl = accelerator.prepare(model, optimizer, train_dl, val_dl)
else:
    model, optimizer, train_dl = accelerator.prepare(model, optimizer, train_dl)

target_speaker_embedding = None
model.train()

for local_epoch in range(NUM_EPOCHS):
    epoch = START_EPOCH + local_epoch
    for step, batch in enumerate(train_dl):
        with accelerator.accumulate(model):
            loss = compute_loss(model, batch)
            accelerator.backward(loss)
            if accelerator.sync_gradients:
                accelerator.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
        if step % 10 == 0:
            accelerator.print(f'Epoch {epoch} | Step {step:4d} | Loss: {loss.item():.4f}')

    if val_dl and (local_epoch + 1) % EVAL_EVERY == 0:
        vl = evaluate(model, val_dl, accelerator)
        if accelerator.is_main_process and vl:
            accelerator.print(f'Epoch {epoch} | Val Loss: {vl:.4f}')

    if accelerator.is_main_process and (local_epoch + 1) % SAVE_EVERY == 0:
        ckpt = os.path.join(OUTPUT_MODEL_PATH, f'checkpoint-epoch-{epoch}')
        os.makedirs(ckpt, exist_ok=True)
        accelerator.unwrap_model(model).save_pretrained(ckpt, safe_serialization=True)

        with open(os.path.join(INIT_MODEL_PATH, 'config.json'), 'r', encoding='utf-8') as f:
            cfg = json.load(f)
        cfg['tts_model_type'] = 'custom_voice'
        tc = cfg.get('talker_config', {})
        tc.setdefault('spk_id', {})[SPEAKER_NAME]        = 3000
        tc.setdefault('spk_is_dialect', {})[SPEAKER_NAME] = False
        cfg['talker_config'] = tc
        with open(os.path.join(ckpt, 'config.json'), 'w', encoding='utf-8') as f:
            json.dump(cfg, f, indent=2, ensure_ascii=False)

        if target_speaker_embedding is not None:
            save_file({'target_speaker_embedding': target_speaker_embedding[0]},
                      os.path.join(ckpt, 'speaker_embedding.safetensors'))
        print(f'✅ Saved → {ckpt}')

print('\n🎉 Training complete!')

## Step 9 — Inference Test 🔊

In [ ]:
import soundfile as sf
import IPython.display as ipd

CHECKPOINT_DIR = os.path.join(OUTPUT_MODEL_PATH, f'checkpoint-epoch-{START_EPOCH + NUM_EPOCHS - 1}')
REF_AUDIO      = sorted(f'{WORK_DIR}/malayalam_data/wavs/{x}'
                        for x in os.listdir(f'{WORK_DIR}/malayalam_data/wavs')
                        if x.endswith('.wav'))[0]
TEST_TEXT  = 'ഈ ഒരു മലയാള ടെക്സ്ററ്റ് ടു സ്പീച്ച് സിസ്റ്റം ആണ്.'
OUTPUT_WAV = '/content/test_malayalam.wav'

base_inf = Qwen3TTSModel.from_pretrained(
    INIT_MODEL_PATH, torch_dtype=torch.bfloat16, attn_implementation=ATTN_IMPLEMENTATION
)
base_inf.model = PeftModel.from_pretrained(base_inf.model, CHECKPOINT_DIR)

wavs, sr = base_inf.generate_voice_clone(text=TEST_TEXT, ref_audio=REF_AUDIO,
        language="Malayalam", x_vector_only_mode=True)
sf.write(OUTPUT_WAV, wavs[0], sr)
print(f'✅ Saved to {OUTPUT_WAV}')
ipd.Audio(OUTPUT_WAV)

## Step 10 — Download Checkpoint (optional)

In [ ]:
CKPT = os.path.join(OUTPUT_MODEL_PATH, f'checkpoint-epoch-{START_EPOCH + NUM_EPOCHS - 1}')
ZIP  = '/content/malayalam_lora.zip'
!zip -r {ZIP} {CKPT}
from google.colab import files
files.download(ZIP)